# TPU v5e-1: runtime recovery + neural-network training probe

این نوت‌بوک برای بررسی محیط TPU کولب و سازگاری forward/backward شبکه عصبی است؛ آموزش PPO اصلی را اجرا یا تغییر نمی‌دهد.

**مهم:** اگر `torch_tpu` نصب نباشد، سلول اول آن را نصب می‌کند و متوقف می‌شود تا Runtime را Restart کنی. بعد از Restart، از ابتدای نوت‌بوک دوباره Run all کن.

پیکربندی آموزش اصلی دست‌نخورده می‌ماند: یک مدل مشترک، ۴۰ محیط موازی، `N_STEPS=1000`، `BATCH_SIZE=4000` و هدف ۴۰٬۰۰۰٬۰۰۰ انتقال محیطی.

## 1. بررسی و آماده‌سازی بک‌اند

قبل از اجرا از **Runtime → Change runtime type** گزینهٔ **TPU v5e-1** را انتخاب کن.

In [ ]:
import os, sys, platform, importlib.util, subprocess
print('Python:', sys.version)
print('Platform:', platform.platform())
print('COLAB_TPU_ADDR set:', bool(os.environ.get('COLAB_TPU_ADDR')))
print('PyTorch module:', importlib.util.find_spec('torch'))
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
spec = importlib.util.find_spec('torch_tpu')
print('torch_tpu module:', spec)
if spec is None:
    print('\nInstalling official TorchTPU package...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_tpu==0.2.0', '--extra-index-url', 'https://download.pytorch.org/whl/cpu'])
    print('\nINSTALL COMPLETE. Now use Runtime → Restart session, then run this notebook again from the first cell.')
    raise SystemExit('Restart runtime after installing TorchTPU, then rerun from the top.')
else:
    print('torch_tpu already installed; continuing.')

## 2. تست واقعی TPU و گرادیان

این سلول محاسبه روی TPU، انتقال نتیجه به CPU و پس‌انتشار گرادیان را بررسی می‌کند.

In [ ]:
import torch, traceback
try:
    import torch_tpu
    print('PyTorch:', torch.__version__)
    print('TorchTPU import: OK')
    print('torch accelerator available:', torch.accelerator.is_available() if hasattr(torch, 'accelerator') else 'API unavailable')
    device = torch.device('tpu')
    x = torch.randn(256, 256, device=device, requires_grad=True)
    loss = (x @ x.T).mean()
    loss.backward()
    print('Device:', device)
    print('Loss:', loss.detach().cpu().item())
    print('Gradient exists:', x.grad is not None)
    print('Gradient sample:', x.grad.flatten()[0].cpu().item())
    assert x.grad is not None and torch.isfinite(x.grad).all().item(), 'Gradient is missing or non-finite'
    print('\nSUCCESS: TPU forward + backward passed')
except Exception:
    print('\nTPU FORWARD/BACKWARD FAILED:')
    traceback.print_exc()
    raise

## 3. Benchmark کوچک آموزش MLP روی CPU و TPU

این فقط مقایسه محاسبات شبکه عصبی است؛ سرعت کل PPO را اندازه نمی‌گیرد و تضمین نمی‌کند Stable-Baselines3 مستقیماً روی TPU اجرا شود.

In [ ]:
import time
from torch import nn
torch.manual_seed(42)
BATCH, INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM = 256, 64, 128, 4
WARMUP, STEPS = 5, 20
def benchmark(device):
    model = nn.Sequential(nn.Linear(INPUT_DIM, HIDDEN_DIM), nn.Tanh(), nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.Tanh(), nn.Linear(HIDDEN_DIM, OUTPUT_DIM)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)
    obs = torch.randn(BATCH, INPUT_DIM, device=device)
    target = torch.randn(BATCH, OUTPUT_DIM, device=device)
    def step():
        opt.zero_grad(set_to_none=True)
        loss = ((model(obs) - target) ** 2).mean()
        loss.backward()
        opt.step()
        return loss
    last = None
    for _ in range(WARMUP): last = step()
    _ = last.detach().cpu().item()
    start = time.perf_counter()
    for _ in range(STEPS): last = step()
    final_loss = last.detach().cpu().item()
    elapsed = time.perf_counter() - start
    return {'device': str(device), 'seconds': round(elapsed, 3), 'steps_per_second': round(STEPS/elapsed, 2), 'final_loss': round(final_loss, 6)}
cpu_result = benchmark(torch.device('cpu'))
tpu_result = benchmark(torch.device('tpu'))
print('CPU:', cpu_result)
print('TPU:', tpu_result)
print('TPU/CPU throughput ratio:', round(tpu_result['steps_per_second']/cpu_result['steps_per_second'], 2))

## 4. چه چیزی را بفرستی؟

خروجی کامل سلول‌های ۱ تا ۳ را بفرست. اگر نصب انجام شد، Runtime را Restart کن و نوت‌بوک را از بالا دوباره اجرا کن.

حتی اگر MLP روی TPU سریع‌تر بود، قبل از تغییر مسیر PPO باید سازگاری الگوریتم، بافرهای rollout، optimizer و محیط شبیه‌سازی C++ جداگانه بررسی شود. این نوت‌بوک هیچ فایل یا تنظیم تولیدی را تغییر نمی‌دهد.